In [12]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent
sys.path.append(str(ROOT))

import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

from sklearn.manifold import TSNE
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

from model.classes import DEVANAGARI_CLASSES
import onnxruntime as ort

TEST_DIR = "../data/DevanagariHandwrittenCharacterDataset/Test"

IMG_SIZE = (32, 32)
BATCH_SIZE = 32
NUM_CLASSES = 46

In [13]:
transform = transforms.Compose(
    [
        transforms.Grayscale(num_output_channels=1),
        transforms.Resize(IMG_SIZE),
        transforms.ToTensor(),
    ]
)

test_dataset = datasets.ImageFolder(
    TEST_DIR,
    transform=transform,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
)

In [14]:
MODEL_PATH = "../model/hindi_cnn.onnx"

session = ort.InferenceSession(
    MODEL_PATH,
    providers=["CPUExecutionProvider"],
)

input_name = session.get_inputs()[0].name

In [15]:
embeddings = []
labels = []
predictions = []

for images, targets in test_loader:

    # plt.imshow(images[0].squeeze(), cmap="gray")
    # plt.show()
    # torch tensor -> numpy
    images_np = images.numpy().astype(np.float32)

    logits, embedding = session.run(None, {input_name: images_np})
    preds = np.argmax(logits, axis=1)

    embeddings.append(embedding)

    labels.append(targets.numpy())
    # print(DEVANAGARI_CLASSES[logits[0].argmax()])
    predictions.append(preds)


embeddings = np.concatenate(embeddings)
labels = np.concatenate(labels)
predictions = np.concatenate(predictions)


print("Embedding shape:", embeddings.shape)

Embedding shape: (13800, 64)


In [ ]:
tsne = TSNE(
    n_components=2,
    random_state=42,
    perplexity=30,
)

points = tsne.fit_transform(embeddings)

In [17]:
from matplotlib.font_manager import FontProperties

devanagari_font = FontProperties(
    fname="/System/Library/Fonts/Supplemental/DevanagariMT.ttc"
)

plt.figure(figsize=(12, 10))

scatter = plt.scatter(
    points[:, 0],
    points[:, 1],
    c=labels,
    cmap="tab20",
    s=8,
)


for class_id in range(NUM_CLASSES):

    class_points = points[labels == class_id]

    if len(class_points) == 0:
        continue

    centroid = class_points.mean(axis=0)

    plt.text(
        centroid[0],
        centroid[1],
        DEVANAGARI_CLASSES[class_id],
        fontsize=14,
        fontproperties=devanagari_font,
    )


plt.title("Devanagari CNN Embedding Space (t-SNE)")
plt.xlabel("t-SNE 1")
plt.ylabel("t-SNE 2")

plt.savefig(
    "embedding_tsne.png",
    dpi=300,
    bbox_inches="tight",
)

plt.close()

In [18]:
cm = confusion_matrix(
    labels,
    predictions,
)


display = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=DEVANAGARI_CLASSES,
)

fig, ax = plt.subplots(figsize=(14, 14))

display.plot(
    ax=ax,
    xticks_rotation=90,
    cmap="Blues",
)

from matplotlib.font_manager import FontProperties

devanagari_font = FontProperties(
    fname="/System/Library/Fonts/Supplemental/DevanagariMT.ttc"
)

for label in ax.get_xticklabels():
    label.set_fontproperties(devanagari_font)

for label in ax.get_yticklabels():
    label.set_fontproperties(devanagari_font)

plt.title("Devanagari CNN Confusion Matrix")

plt.savefig(
    "confusion_matrix.png",
    dpi=300,
    bbox_inches="tight",
)

plt.close()